***Coding Question: Loan Approval with a Decision Tree***

Column	Meaning
Age	Applicant age
Income_k	Annual income in thousands
CreditScore	300 to 850
ExistingDebt_k	Current debt in thousands
Approved	Target: 1 = approved, 0 = rejected

The data is synthetic, with about 8% random noise added on purpose, like real life.

Your tasks

Task 1: Explore. Load the CSV, print the shape and the count of each class in Approved.

Task 2: Split. Make an 80/20 train/test split with random_state=42 and stratify=y.

Task 3: Overfitting experiment. Train DecisionTreeClassifier(criterion="gini", random_state=42) with max_depth in [None, 2, 3, 4]. Print the train and test accuracy for each.

Task 4: Final model. Use max_depth=3. Print the tree as text with export_text, then plot it with plot_tree and find the root’s gini value. Print the confusion matrix on the test set.

Task 5: Unseen applicants. Predict approval and probability for:

Applicant	Age	Income_k	CreditScore	ExistingDebt_k
A	30	80	720	10
B	45	35	600	40
C	28	60	580	15

In [92]:
import pandas as pd

df=pd.read_csv('loan_data.csv')
cdf=df

In [93]:
cdf.head()

,Age,Income_k,CreditScore,ExistingDebt_k,Approved
0,24,56,849,36,1
1,51,111,728,46,1
2,47,70,476,44,0
3,38,90,835,8,1
4,38,66,575,49,0


In [94]:
cdf.shape

(200, 5)

In [95]:
X=cdf.drop(['Approved'],axis=1)
Y=cdf['Approved']

In [96]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,stratify=Y,random_state=42)

tree=DecisionTreeClassifier()
tree.fit(x_train,y_train)


,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [97]:
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
y_pred=tree.predict(x_test)
print("Accuracy :",accuracy_score(y_test,y_pred))
print("f1-score :",f1_score(y_test,y_pred))
print('confusion Matrix :',confusion_matrix(y_test,y_pred))
print('classification Report :',classification_report(y_test,y_pred))


Accuracy : 0.775
f1-score : 0.6896551724137931
confusion Matrix : [[21  4]
 [ 5 10]]
classification Report :               precision    recall  f1-score   support

           0       0.81      0.84      0.82        25
           1       0.71      0.67      0.69        15

    accuracy                           0.78        40
   macro avg       0.76      0.75      0.76        40
weighted avg       0.77      0.78      0.77        40



***Prediction on new data*** 

In [98]:
newdata=pd.DataFrame({
    'Age':[30],
    'Income_k':[80],
    'CreditScore':[720],
    'ExistingDebt_k':[10],   
})

In [99]:
new_data_prediction=tree.predict(newdata)
print("Prediction on unseen data expected is 1 ie loan Approved Got :",new_data_prediction)

Prediction on unseen data expected is 1 ie loan Approved Got : [1]


In [100]:
new_applicants = pd.DataFrame(
    [[30, 80, 720, 10],
     [45, 35, 600, 40],
     [28, 60, 580, 15]],
    columns=["Age", "Income_k", "CreditScore", "ExistingDebt_k"]
)

pred  = tree.predict(new_applicants)
proba = tree.predict_proba(new_applicants)

for name, p, pr in zip(["A", "B", "C"], pred, proba):
    label = "APPROVE" if p == 1 else "REJECT"
    print(f"Applicant {name}: {label}  (P(reject)={pr[0]:.2f}, P(approve)={pr[1]:.2f})")

Applicant A: APPROVE  (P(reject)=0.00, P(approve)=1.00)
Applicant B: REJECT  (P(reject)=1.00, P(approve)=0.00)
Applicant C: REJECT  (P(reject)=1.00, P(approve)=0.00)


***USING GRID SEARCH CV***

In [101]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

def Dtree_mod():
    return Pipeline([
        ('Dec_tree',DecisionTreeClassifier())
    ])

mod=Dtree_mod()

In [102]:
param_grid={
    'Dec_tree__max_depth':list(range(1,4)),
}

grid=GridSearchCV(
    estimator=mod,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy'
)

grid.fit(x_train,y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'Dec_tree__max_depth': [1, 2, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [109]:

print("Accuracy of Model Before Tunning  :",accuracy_score(y_test,y_pred))
print("Confusion Matrix of Base model :",confusion_matrix(y_test,y_pred))
print("classification Report ",classification_report(y_test,y_pred))

print("---------------------------------------------------")
print("\n---------------MODEL TUNNING ----------------------\n")
best_model=grid.best_estimator_
print("Accuracy Of Model After Tunning Best CV Score :",grid.best_score_)
y_pred = best_model.predict(x_test)
print("best Param :",grid.best_params_)
print("best model :",grid.best_estimator_)
print("\nconfusion matrix :")
print(confusion_matrix(y_test,y_pred))
print("\nClassification Report :")
print(classification_report(y_test,y_pred))

Accuracy of Model Before Tunning  : 0.825
Confusion Matrix of Base model : [[22  3]
 [ 4 11]]
classification Report                precision    recall  f1-score   support

           0       0.85      0.88      0.86        25
           1       0.79      0.73      0.76        15

    accuracy                           0.82        40
   macro avg       0.82      0.81      0.81        40
weighted avg       0.82      0.82      0.82        40

---------------------------------------------------

---------------MODEL TUNNING ----------------------

Accuracy Of Model After Tunning Best CV Score : 0.86875
best Param : {'Dec_tree__max_depth': 3}
best model : Pipeline(steps=[('Dec_tree', DecisionTreeClassifier(max_depth=3))])

confusion matrix :
[[22  3]
 [ 4 11]]

Classification Report :
              precision    recall  f1-score   support

           0       0.85      0.88      0.86        25
           1       0.79      0.73      0.76        15

    accuracy                           0.82  

***Prediction By BASE MODEL***

In [110]:
pred  = tree.predict(new_applicants)
proba = tree.predict_proba(new_applicants)

for name, p, pr in zip(["A", "B", "C"], pred, proba):
    label = "APPROVE" if p == 1 else "REJECT"
    print(f"Applicant {name}: {label}  (P(reject)={pr[0]:.2f}, P(approve)={pr[1]:.2f})")

Applicant A: APPROVE  (P(reject)=0.00, P(approve)=1.00)
Applicant B: REJECT  (P(reject)=1.00, P(approve)=0.00)
Applicant C: REJECT  (P(reject)=1.00, P(approve)=0.00)


HERE AS PROBABILITES ARE Either 0 or 1 therefore the decison tree is keep on growing as Max depth is not assigned to the tree therefore it is giving very accurate result thats why this model is as Overfit model 


BasELINE MODEL SHOWS OVERFITTING

***Predicion By BEST MODEL***

In [111]:
pred  = best_model.predict(new_applicants)
proba = best_model.predict_proba(new_applicants)

for name, p, pr in zip(["A", "B", "C"], pred, proba):
    label = "APPROVE" if p == 1 else "REJECT"
    print(f"Applicant {name}: {label}  (P(reject)={pr[0]:.2f}, P(approve)={pr[1]:.2f})")

Applicant A: APPROVE  (P(reject)=0.03, P(approve)=0.97)
Applicant B: REJECT  (P(reject)=0.96, P(approve)=0.04)
Applicant C: APPROVE  (P(reject)=0.12, P(approve)=0.88)
